# Bibliometry of one person with OpenAlex (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/celoy/bibliometry/blob/main/notebooks/colab_one_person.ipynb)

Enter **one name** and an institution: this notebook finds the person's OpenAlex profile(s), downloads all their publications (all years by default), and reports:

* the publications **signed with the institution** — the list, with every OpenAlex field;
* how many are in the world's **top 10% and 1% most cited** of their field and year, and which ones;
* the **journals** used, with their 2-year mean citedness and whether they rank in the world top 10% of their subfield.

Run the cells from top to bottom. Results are saved in your Google Drive, in `My Drive/bibliometry/people/<name>/`.

### Before you start: your OpenAlex API key (2 minutes, once)

1. Create a free account at [openalex.org](https://openalex.org) (*Sign up*, top right).
2. Open [openalex.org/settings/api](https://openalex.org/settings/api) and copy your **API key**.
3. In this notebook, click the **🔑 key icon** in the left sidebar (*Secrets*) → **Add new secret**: name `OPENALEX_API_KEY`, value = your key, and switch on **Notebook access**.

Without a key the notebook still runs, but the first run may exhaust the small anonymous budget (it downloads a one-off reference of ~70,000 journals).

## 1 · Install and connect your Google Drive

In [ ]:
!pip install -q "git+https://github.com/celoy/bibliometry"

from bibliometry.env import project_root

BASE = project_root()  # My Drive/bibliometry

## 2 · Who and where

* **NAME**: first and last name as usually written (e.g. *Ada Lovelace*), or *Last, First*.
* **ORCID** (optional but recommended): e.g. `0000-0002-1825-0097`.
* **INSTITUTION**: a name, an OpenAlex id (`I…`) or a ROR id. The candidates found are listed; if the wrong one is selected, change **INSTITUTION_CHOICE** and run again.
* **ALL_YEARS**: untick to use the dates below.

In [ ]:
EMAIL = ""  # @param {type:"string"}
NAME = ""  # @param {type:"string"}
ORCID = ""  # @param {type:"string"}
INSTITUTION = ""  # @param {type:"string"}
INSTITUTION_CHOICE = 1  # @param {type:"integer"}
ALL_YEARS = True  # @param {type:"boolean"}
FROM_DATE = "2020-01-01"  # @param {type:"date"}
TO_DATE = "2024-12-31"  # @param {type:"date"}

from datetime import date

from bibliometry import pipeline
from bibliometry.config import load_api_key
from bibliometry.institution import describe_institution, find_institutions
from bibliometry.openalex import OpenAlexClient

api_key = load_api_key(BASE)
print("API key:", "found" if api_key else "NOT FOUND — see 'Before you start' above")
with OpenAlexClient(mailto=EMAIL, api_key=api_key) as c:
    candidates = find_institutions(c, INSTITUTION)
for i, record in enumerate(candidates, start=1):
    print(("→ " if i == INSTITUTION_CHOICE else "  ") + f"{i}. {describe_institution(record)}")
dates = ("1900-01-01", date.today().isoformat()) if ALL_YEARS else (FROM_DATE, TO_DATE)
config = pipeline.person_config(BASE, EMAIL, candidates[INSTITUTION_CHOICE - 1], NAME, ORCID, *dates)
config.validate()
print(f"\n{config.roster.person} at {config.institution.name}, {dates[0]} → {dates[1]}")
print(f"Results will be in {config.root}")

## 3 · Find the person's OpenAlex profile(s)

Publications of the institution with an author of that surname are downloaded first; they show which OpenAlex profiles sign with the institution under that name.

In [ ]:
client = pipeline.make_client(config)
pipeline.harvest_institution_step(config, client)
matches, people = pipeline.match_step(config, client)
matches[["author_id", "author_name", "accepted", "confidence", "sources", "orcid_status",
         "works_in_period_with_institution", "works_total", "last_known_institution",
         "top_topics", "openalex_url"]]

`accepted = True` profiles are used. If one is wrong, or one is missing its `True`, open its `openalex_url`, then write your verdict below (`"ok"`, `"wrong"`, or `"conflated"` when OpenAlex mixed this person with a namesake) and run the cell.

In [ ]:
verdicts = {
    # "A1234567890": "ok",
}
if verdicts:
    pipeline.set_verdicts(config, {(people.person_id.iloc[0], a): v for a, v in verdicts.items()})
    matches, people = pipeline.match_step(config, None)
people[["person_id", "author_ids", "match_status"]]

## 4 · Download all publications and compute the indicators

The first run on a Drive also downloads the world journal reference (~70,000 journals, a few minutes, once).

In [ ]:
pipeline.harvest_people_step(config, client)
tables = pipeline.tables_step(config)
out = pipeline.lists_step(config, client)

## 5 · Results

In [ ]:
from bibliometry import analysis as A

works = tables["works"]
own = works[works.roster_people != ""]
signed = own[own.in_list_roster]
print(f"{len(own)} publications found, {len(signed)} signed with {config.institution.name} "
      f"({signed.publication_year.min()}–{signed.publication_year.max()})")
per_year = own.assign(
    signed=own.in_list_roster.map({True: "signed with the institution", False: "signed elsewhere"})
).groupby(["publication_year", "signed"]).size().unstack(fill_value=0)
fig = A.plot_lines(per_year, "Publications per year")

### Top 10% and 1% most cited (publications signed with the institution)

Compared with publications of the same field, year and type worldwide; an average researcher would have 10% and 1%. Recent years are less reliable.

In [ ]:
cited = out["top_cited"]
cited.style.format({"coverage": "{:.0%}", "share_top10": "{:.1%}", "share_top1": "{:.1%}"})

In [ ]:
out["top1_papers_list2"]

### Journals

`world_pct_2yr_mean_citedness` = share of the world's active journals of the same subfield with a higher 2-year mean citedness (0.05 = better than 95%); `world_top10_…` is true below 0.10.

In [ ]:
v = out["venues"]
v[v.ranked][["venue_name", "main_subfield", "works_list2", "2yr_mean_citedness",
             "world_pct_2yr_mean_citedness", "world_top10_2yr_mean_citedness", "world_n"]]

In [ ]:
out["top_venues"].T

### Files

All in `My Drive/bibliometry/people/<name>/data/lists/`: `list2_roster.xlsx` (the signed publications, every OpenAlex field), `venues.xlsx`, `indicators.xlsx`. The columns are described in [docs/columns.md](https://github.com/celoy/bibliometry/blob/main/docs/columns.md).

In [ ]:
from bibliometry.env import offer_download

for name in ("list2_roster.xlsx", "indicators.xlsx"):
    offer_download(config.data_path / "lists" / name)